# SSQ Quant Research System

Kaggle 一键运行：从 GitHub 加载源码，通过公开接口自动获取双色球历史开奖数据，先运行测试和随机基线，再按开关运行 LSTM。

本 Notebook 只做研究与回测，不承诺或证明可以预测中奖号码。

In [ ]:
# ===== 用户配置 =====
RUN_LSTM = False
WINDOW = 30
SEED = 42
LSTM_EPOCHS = 20
REPO_URL = 'https://github.com/wangmou23/ssq-quant-research-system.git'
REPO_DIR = '/kaggle/working/ssq-quant-research-system'
DATA_CSV = None  # 正常情况保持 None：Notebook 会自动取数
print('配置完成；自动取数已启用。')

In [ ]:
# ===== 加载 GitHub 源码 =====
import os, sys, subprocess, pathlib, shutil
repo_dir = pathlib.Path(REPO_DIR)
if repo_dir.exists() and not (repo_dir / 'ssq_research').exists():
    shutil.rmtree(repo_dir)
if not repo_dir.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo_dir)], check=True)
sys.path.insert(0, str(repo_dir))
print('源码目录:', repo_dir)
print('提交:', subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
# ===== 环境检查：不执行 pip install -e，避免 Kaggle Python 3.13 构建失败 =====
import platform, importlib.util
print('Python:', platform.python_version())
print('pytest:', bool(importlib.util.find_spec('pytest')))
try:
    import tensorflow as tf
    print('TensorFlow:', tf.__version__)
    print('GPU:', tf.config.list_physical_devices('GPU'))
except Exception as exc:
    print('TensorFlow 不可用；基线仍可运行。', repr(exc))

In [ ]:
# ===== 项目测试 =====
test_env = os.environ.copy()
test_env['PYTHONPATH'] = str(repo_dir) + os.pathsep + test_env.get('PYTHONPATH', '')
test_result = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], cwd=str(repo_dir), env=test_env, text=True)
if test_result.returncode != 0:
    raise RuntimeError('项目测试失败，请先查看上面的 pytest 输出。')
print('项目测试通过。')

In [ ]:
# ===== 自动获取双色球历史数据 =====
from datetime import datetime, timezone
from ssq_research.data_fetch import fetch_ssq_draws, DEFAULT_SSQ_URL
from ssq_research.core import load_csv
data_source = DEFAULT_SSQ_URL
try:
    real_draws = fetch_ssq_draws(DEFAULT_SSQ_URL, timeout=30)
except Exception as exc:
    print('自动接口访问失败:', repr(exc))
    if DATA_CSV:
        real_draws = load_csv(pathlib.Path(DATA_CSV))
        data_source = DATA_CSV
        print('已使用备用 CSV:', DATA_CSV)
    else:
        raise RuntimeError('自动取数失败。请确认 Kaggle Internet=On；不需要上传数据，若接口暂时不可用再设置 DATA_CSV。') from exc
print('数据源:', data_source)
print('抓取时间 UTC:', datetime.now(timezone.utc).isoformat())
print('历史期数:', len(real_draws))
print('范围:', real_draws[0].issue, '→', real_draws[-1].issue)

In [ ]:
# ===== 合成数据 smoke test =====
from datetime import date, timedelta
from ssq_research.core import Draw, RandomStrategy, FrequencyStrategy, run_backtest, summarize
start = date(2020, 1, 1)
smoke_draws = [Draw(str(2020001+i), (start + timedelta(days=i)).isoformat(), tuple(range(1+i%20, 7+i%20)), 1+i%16) for i in range(80)]
print(summarize(run_backtest(smoke_draws, (RandomStrategy(), FrequencyStrategy()), window=WINDOW, seed=SEED)))

In [ ]:
# ===== 真实历史数据：同成本基线回测 =====
baseline_rows = run_backtest(real_draws, (RandomStrategy(), FrequencyStrategy()), window=WINDOW, seed=SEED)
baseline_summary = summarize(baseline_rows)
baseline_summary

In [ ]:
# ===== 可选 LSTM 时间顺序回测 =====
if RUN_LSTM:
    from ssq_research.lstm import LstmConfig, walk_forward_lstm
    config = LstmConfig(window=10, hidden_units=32, layers=2, dropout=0.1, epochs=LSTM_EPOCHS, batch_size=32, retrain_every=10, max_predictions=100, verbose=1)
    lstm_rows = walk_forward_lstm(real_draws, config, min_history=max(WINDOW, config.window + 1), seed=SEED)
    print('LSTM 评估期数:', len(lstm_rows))
    print(summarize(lstm_rows))
else:
    print('RUN_LSTM=False：已完成自动取数和基线回测；确认基线后再改为 True。')

In [ ]:
# ===== 公平对照表：同一最后 N 期、同成本 =====
from ssq_research.core import RandomStrategy, FrequencyStrategy, run_backtest, summarize
COMPARISON_PERIODS = 100
all_baseline_rows = run_backtest(real_draws, (RandomStrategy(), FrequencyStrategy()), window=WINDOW, seed=SEED)
baseline_target_issues = {d.issue for d in sorted(real_draws, key=lambda d: (d.draw_date, d.issue))[-COMPARISON_PERIODS:]}
fair_baseline_rows = [r for r in all_baseline_rows if r['issue'] in baseline_target_issues]
fair_rows = fair_baseline_rows + (lstm_rows if RUN_LSTM else [])
fair_comparison = summarize(fair_rows)
print(f'公平评估区间：最后 {COMPARISON_PERIODS} 期；每期 1 注；每注 2 元')
fair_comparison


In [ ]:
# ===== 最终模型：使用全部历史数据生成 5 组下一期研究性候选 =====
if RUN_LSTM:
    import numpy as np
    from ssq_research.lstm import LstmConfig, train_model, encode_draw
    N_COMBINATIONS = 5
    BUDGET = 30
    final_config = LstmConfig(window=30, hidden_units=32, layers=2, dropout=0.1, epochs=LSTM_EPOCHS, batch_size=32, verbose=1)
    final_model = train_model(real_draws, final_config, validation_split=0.0)
    recent = np.expand_dims(np.stack([encode_draw(d) for d in real_draws[-final_config.window:]]), 0)
    probabilities = final_model.predict(recent, verbose=0)[0]
    red_scores = probabilities[:33]
    blue_scores = probabilities[33:]
    red_pool = (np.argsort(red_scores)[::-1][:10] + 1).tolist()
    blue_pool = (np.argsort(blue_scores)[::-1][:max(5, N_COMBINATIONS)] + 1).tolist()
    candidates = []
    for i in range(N_COMBINATIONS):
        # 轮换红球池，减少 5 组之间完全重复；蓝球覆盖 Top 5
        ranked_reds = sorted(red_pool, key=lambda n: (-float(red_scores[n - 1]), (n + i) % 3))
        reds = tuple(sorted(ranked_reds[i % 5:i % 5 + 6]))
        if len(reds) < 6:
            reds = tuple(sorted((red_pool + red_pool)[i:i + 6]))
        candidates.append((reds, blue_pool[i]))
    print('训练数据:', real_draws[0].issue, '→', real_draws[-1].issue)
    print(f'候选组数: {N_COMBINATIONS}；预算上限: {BUDGET} 元；当前 5 注成本: {N_COMBINATIONS * 2} 元')
    print('红球概率排名 Top 10:', [(int(i + 1), round(float(red_scores[i]), 4)) for i in np.argsort(red_scores)[::-1][:10]])
    print('蓝球概率排名 Top 5:', [(int(i + 1), round(float(blue_scores[i]), 4)) for i in np.argsort(blue_scores)[::-1][:5]])
    print('5 组研究性候选组合:')
    for number, (reds, blue) in enumerate(candidates, start=1):
        print(f'{number}. 红球 {reds} + 蓝球 {blue:02d}')
    print('声明：这是历史样本上的模型输出，不代表中奖概率得到保证。')
else:
    print('RUN_LSTM=False：设置为 True 后运行最终模型预测。')


## 结果解释

- 必须保留数据源、抓取时间、期数、窗口、随机种子和训练配置。
- 只比较同成本、同票数的随机基线与策略。
- 单次命中、最高奖级或历史曲线不能证明未来存在优势。
- 如果自动接口失败，优先检查 Kaggle `Internet: On`；不要运行旧版 Notebook 中的 `pip install -e`。